# 02 — POST periódico (fonte do dataset) — documentação da disciplina

**Projeto:** Preditor de Falhas ML (Grupo 16)
**Disciplinas:** ED2 + Redes + APS
**Card:** S1.6 — POST: 6 medições (fonte do dataset)

> Este notebook é a **documentação / demonstração** pedida pela professora.
> **Não** é o pipeline de produção nem o collector.
> O **core** do POST fica no pacote Python: `create_periodic_measurements`
> (`src/preditor_de_falhas_ml/atlas.py`). A CLI `createPeriodic` chama a mesma
> função — não este notebook.
> O collector (`getResults` / Lambda S1.7) é **só GET** e **não** chama este POST.
>
> POST (uma vez / ao reconfigurar) → 6 `msm_id` periódicos = **fonte**.
> Dataset de treino = **acumulado dos GETs** nesses IDs (S1.2 / S1.7).
> Runbook e tabela de IDs: `docs/dataset-fonte-atlas.md`.

## Equipe

| Integrante | Papel nesta entrega |
|---|---|
| Guilherme Leite Tavares | [preencher] |
| Alexandre Tiago de Oliveira | [preencher] |
| Ingrid Ferreira de Sousa | [preencher] |
| Kauan Garcia Dias de Oliveira | [preencher] |
| Lucas Eduardo Malachias Bagatela | [preencher] |
| Stephanie Vitoria Bessa dos Santos | [preencher] |

## Decisões (justificativa)

| Decisão | Escolha | Por quê |
|---|---|---|
| Verbo HTTP | **POST** `/measurements/` periódico | Aponta aos destinos do hub; cria a **fonte** do dataset |
| Onde está o código | Pacote `src/preditor_de_falhas_ml/atlas.py` | `create_periodic_measurements` + CLI `createPeriodic` |
| Papel deste notebook | Documentação da disciplina + demo pontual | Exigência da professora; **não** é o collector nem a Lambda |
| One-off vs periódico | `is_oneoff: false`, `interval: 900` | One-off (`getData`) custa mais e **não** serve para a série de treino |
| Destinos | `94.140.14.14` (AdGuard), `208.67.222.222` (OpenDNS), `202.12.28.131` (APNIC) | Intenção original (`8.8.8.8`, `1.1.1.1`, `202.12.27.33`) superseded-for-quota; retry só se a cota global liberar |
| Collector | Só GET (`fetch_measurement_results` / `getResults`) | POST no schedule = IDs novos, sem série estável |
| Dataset de treino | Acumulado dos GETs nos 6 `msm_id` | POST é só setup; curated vem depois (S1.7+) |
| Persistência dos IDs | `write_measurement_ids` em `data/` (gitignorado) | Sem API key no git; tabela no runbook |

## Params (matriz do hub)

Os mesmos da função do pacote e da CLI `createPeriodic` — **não** editar destinos aqui.

| Destino | Papel | Tipo | Pacotes | Probes | Intervalo |
|---|---|---|---|---|---|
| 94.140.14.14 (AdGuard DNS) | estável | ping | 5 | 2 BR | 900 s |
| 94.140.14.14 (AdGuard DNS) | estável | traceroute ICMP | 3 | 2 BR | 900 s |
| 208.67.222.222 (OpenDNS) | estável | ping | 5 | 2 BR | 900 s |
| 208.67.222.222 (OpenDNS) | estável | traceroute ICMP | 3 | 2 BR | 900 s |
| 202.12.28.131 (APNIC) | caminho longo | ping | 5 | 2 BR | 900 s |
| 202.12.28.131 (APNIC) | caminho longo | traceroute ICMP | 3 | 2 BR | 900 s |

`is_oneoff: false`. Seis medições periódicas. Continuam cobrando créditos até serem paradas no Atlas (~210 / 15 min → ~20 mil/dia).

Intenção original (superseded-for-quota; **não** entra neste POST): `8.8.8.8`, `1.1.1.1`, `202.12.27.33`. Retry só se a cota global liberar.

Na raiz do repo: `uv sync`.
`RIPE_ATLAS_API_KEY` no ambiente ou `.env` (**não** commitado).
Caminho preferido ao vivo: CLI no runbook `docs/dataset-fonte-atlas.md`.

```bash
uv run --env-file .env python -m preditor_de_falhas_ml createPeriodic --ids-file data/msm_ids.json
```

A célula ao vivo abaixo fica **desligada** (`CRIAR_MEDICOES = False`). Só ligue se for intencional.

In [ ]:
import os
from pathlib import Path

from preditor_de_falhas_ml import (
    create_periodic_measurements,
    get_credits,
    write_measurement_ids,
)
from preditor_de_falhas_ml.atlas import (
    HUB_COUNTRY_CODE,
    HUB_INTERVAL_SECONDS,
    HUB_PROBE_COUNT,
    HUB_SPECS,
)

# True só se for intencional: consome créditos e cria 6 medições periódicas
# (correm até serem paradas no Atlas). Preferir a CLI createPeriodic.
CRIAR_MEDICOES = False
IDS_PATH = Path("data/msm_ids.json")

api_key = os.environ.get("RIPE_ATLAS_API_KEY", "")

{
    "n_defs": len(HUB_SPECS),
    "interval": HUB_INTERVAL_SECONDS,
    "is_oneoff": False,
    "probes": f"{HUB_PROBE_COUNT} {HUB_COUNTRY_CODE}",
    "CRIAR_MEDICOES": CRIAR_MEDICOES,
    "ids_path": str(IDS_PATH),
    "tem_chave": bool(api_key),
    "matriz": [
        {
            "target": spec.target,
            "role": spec.role,
            "type": spec.measurement_type,
            "packets": spec.packets,
        }
        for spec in HUB_SPECS
    ],
}

## Chamada ao core (POST, gated)

A célula abaixo **importa** `create_periodic_measurements` e `get_credits` do pacote.
Não usa `requests` nas células e **não** chama `get_data` (one-off / demo — não é a série de treino).

- Sem `RIPE_ATLAS_API_KEY`: mensagem de bloqueio; **não inventa** `msm_id`.
- Com chave e `CRIAR_MEDICOES = False` (padrão): não faz POST; documenta o runbook.
- Com chave e `CRIAR_MEDICOES = True`: **consome créditos** e cria 6 medições periódicas. Só rode se for intencional.

In [ ]:
msm_ids = []
credits_before = None
credits_after = None

if not api_key:
    print(
        "POST ao vivo bloqueado: RIPE_ATLAS_API_KEY ausente "
        "(ambiente ou .env, não commitado). Nenhum msm_id inventado. "
        "Ver docs/dataset-fonte-atlas.md."
    )
elif not CRIAR_MEDICOES:
    print(
        "Chave presente, mas CRIAR_MEDICOES=False. "
        "Não vou chamar create_periodic_measurements. "
        "Caminho preferido: CLI createPeriodic (docs/dataset-fonte-atlas.md). "
        "Só mude CRIAR_MEDICOES para True se for intencional — "
        "consome créditos e as 6 medições ficam periódicas até parar no Atlas."
    )
else:
    print(
        "ATENÇÃO: POST das 6 medições periódicas do hub. "
        "Consome créditos (~210 / 15 min) até serem paradas no Atlas."
    )
    credits_before = get_credits(api_key)
    msm_ids = create_periodic_measurements(api_key)
    credits_after = get_credits(api_key)

{
    "msm_ids": msm_ids,
    "n": len(msm_ids),
    "credits_before": credits_before,
    "credits_after": credits_after,
    "inventou_id": False,
}

## Persistência dos msm_id (sem a API key)

`write_measurement_ids` só grava se a célula anterior devolveu IDs reais.
O arquivo fica em `data/` (gitignorado). **Nunca** escreve a API key.
Copiar os IDs para `RIPE_ATLAS_MSM_IDS` (Secret/SSM) e para a tabela em `docs/dataset-fonte-atlas.md`.

In [ ]:
if not msm_ids:
    print(
        "Nada a persistir: sem msm_id real "
        "(chave ausente ou CRIAR_MEDICOES=False). "
        "Não inventar IDs. Tabela do runbook: docs/dataset-fonte-atlas.md."
    )
else:
    written = write_measurement_ids(msm_ids, output_path=IDS_PATH)
    payload = written.read_text(encoding="utf-8")
    if api_key:
        assert api_key not in payload
    {
        "ids_file": str(written),
        "msm_ids": msm_ids,
        "export": "export RIPE_ATLAS_MSM_IDS=" + ",".join(str(i) for i in msm_ids),
    }

## Dataset = GETs (não o payload do POST)

Depois dos 6 `msm_id` existirem, o histórico de treino vem de `fetch_measurement_results` / CLI `getResults` (notebook `01_coleta_atlas_raw.ipynb`, depois Lambda S1.7). Este notebook **não** reimplementa o GET e **não** rotula `status_real`.

Validar 1 ciclo (~15 min) pela CLI:

```bash
uv run --env-file .env python -m preditor_de_falhas_ml createPeriodic --ids-file data/msm_ids.json --wait-seconds 900
```

ou, com IDs já gravados, `getResults --msm-id … --start … --stop …`.
O primeiro ciclo pode estar vazio se as probes BR ainda não reportaram.

## Checklist

- [ ] Equipe preenchida (papéis)
- [ ] Decisões justificadas (POST periódico vs GET; core = pacote; notebook = doc)
- [ ] POST via `from preditor_de_falhas_ml import create_periodic_measurements`
- [ ] Sem `requests` / sem `get_data` como caminho de treino nas células
- [ ] Matriz do hub: 3 destinos × ping + traceroute ICMP, 2 BR, `is_oneoff: false`, `interval` 900
- [ ] Sem chave: mensagem de bloqueio; **nenhum** `msm_id` inventado
- [ ] `CRIAR_MEDICOES` padrão `False`; POST ao vivo só se for intencional (créditos)
- [ ] IDs via `write_measurement_ids` em `data/` (sem a API key)
- [ ] Claro para a banca: **POST = setup**; **dataset = GETs acumulados**; **produção = pacote + Lambda GET**